In [ ]:
# Edge Server Code
# This code should be run on the edge server (laptop)

# Load modules
import socket
import pandas as pd

# Create empty dataframe
df = pd.DataFrame(columns=['Temp','Humd','Label'])

# Open socket
s = socket.socket()
print("Socket successfully created")
port = 12345
s.bind(('',port))
print("socket binded to %s" %(port))

# Listen
s.listen(5)
print("socket is listening")
c, addr = s.accept()
print("Got connection from", addr)

# Save readings to dataframe
num_readings = 1000
while num_readings>=0:
    client = c.recv(1024)
    data = client.decode()
    temp = data.split(" ")[0]
    hum = data.split(" ")[1]
    new_data = {
        "Temp": temp,
        "Humd": hum,
        "Label": 1
    }
    df = df._append(new_data, ignore_index = True)
    print(data)

    num_readings -= 1

# Close socket
c.close()

# Save dataframe to csv
df.to_csv("TemperatureHumidityData.csv", index=False)

Socket successfully created
socket binded to 12345
socket is listening
Got connection from ('172.20.10.3', 41704)
22.1 38
22.1 38
22.2 37
22.2 37
22.2 40
22.2 40
22.2 43
22.2 43
22.4 43
22.4 43
22.5 42
22.5 42
22.6 41
22.6 41
22.7 40
22.7 40
22.8 39
22.8 39
23.0 66
23.0 66
23.0 65
23.0 65
23.0 65
23.0 65
23.1 60
23.1 60
23.1 57
23.1 57
23.1 54
23.1 54
23.1 52
23.1 52
23.1 49
23.1 49
23.1 48
23.1 48
23.0 46
23.0 46
23.0 44
23.0 44
23.0 42
23.0 42
23.0 41
23.0 41
23.0 41
23.0 41
23.0 49
23.0 49
23.0 55
23.0 55
23.0 56
23.0 56
23.1 55
23.1 55
23.1 52
23.1 52
23.2 49
23.2 49
23.2 49
23.2 49
23.3 59
23.3 59
23.5 65
23.5 65
23.7 70
23.7 70
24.0 73
24.0 73
24.2 76
24.2 76
24.5 77
24.5 77
24.6 75
24.6 75
24.8 73
24.8 73
24.9 70
24.9 70
25.0 64
25.0 64
25.0 60
25.0 60
25.0 60
25.0 60
25.0 52
25.0 52
25.0 48
25.0 48
25.0 45
25.0 45
25.0 42
25.0 42
25.0 40
25.0 40
25.0 39
25.0 39
24.9 39
24.9 39
24.8 38
24.8 38
24.8 37
24.8 37
24.7 36
24.7 36
24.6 36
24.6 36
24.5 35
24.5 35
24.5 35
24.5 35
24.4 3

In [ ]:
# Raspberry Pi Code
# This should be run on the Raspberry Pi.

# Load Modules
import time
import board
import adafruit_dht
import socket

# Set address and port
ip = "169.254.74.127" # Match edge server
port = 12345

# Connect to DHT11
dht = adafruit_dht.DHT11(board.D4, use_pulseio=False)

# Connect to edge server's socket
s = socket.socket()
s.connect((ip, port))

# Send readings to edge server
while True:
    try:
        temp = dht.temperature
        hum = dht.humidity
        print(f"Temp: {temp:.1f}°C Humidity: {hum:.1f}%")
        data = str(temp) + " " + str(hum)
        s.send(data.encode())
        time.sleep(1)
    except Exception as e:
        print("Error reading DHT11:", e)

# Terminate socket connection
s.close()

In [ ]:
# Create fake data
import pandas as pd
import random
i = 1000 # Amount of fake entries to add

# Add fake entries
while i > 0:
    df = pd.read_csv("data.csv")

    # Creates fake data between 0 to i. Will trend towards 0.
    new_data = {
        'Temp': random.randint(0,i),
        'Humd': random.randint(0,i),
        'Label': 0
    }

    df = df._append(new_data, ignore_index = True)
    df.to_csv("data.csv", index=False)

    i -= 1

In [2]:
# Load Modules
import numpy as np
import pandas as pd
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn import metrics

import pickle

# Load Data
df = pd.read_csv("TemperatureHumidityData.csv")
df.head()

,Temp,Humd,Label
0,22.1,38,1
1,22.1,38,1
2,22.2,37,1
3,22.2,37,1
4,22.2,40,1


In [3]:
# Prepare Data
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

#df = df.drop_duplicates() #remove duplicates

x = df.drop(['Label'],axis=1) #remove class or label
y = df['Label'] #load label

# Split into train and test
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=.2)

# Scale Data
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train) #scale training set
x_test = scaler.transform(x_test)

Missing Values:
 Temp     0
Humd     0
Label    0
dtype: int64


In [4]:
# KNN Model
knn = KNeighborsClassifier(n_neighbors=3)
KNN = knn.fit(x_train,y_train)
knn_predict = KNN.predict(x_test)

print("KNN Accuracy:", metrics.accuracy_score(y_test,knn_predict))

KNN Accuracy: 0.9925187032418953


In [5]:
# Native Bayes Model
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(x_train, y_train)

nb_predict = nb.predict(x_test)
nb_accuracy = metrics.accuracy_score(y_test, nb_predict)

print("Naive Bayes Accuracy:", nb_accuracy)

Naive Bayes Accuracy: 0.9925187032418953


In [6]:
# QDA Model
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
qda = QDA(reg_param=1e-4)
qda.fit(x_train, y_train)

qda_predict = qda.predict(x_test)
qda_accuracy = metrics.accuracy_score(y_test, qda_predict)

print("Quadratic Discriminant Analysis Accuracy:", qda_accuracy)

Quadratic Discriminant Analysis Accuracy: 0.9800498753117207


In [7]:
# LDA Model
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
lda = LDA()
lda.fit(x_train, y_train)

lda_predict = lda.predict(x_test)
lda_accuracy = metrics.accuracy_score(y_test, lda_predict)

print("Linear Discriminant Analysis Accuracy:", lda_accuracy)

Linear Discriminant Analysis Accuracy: 0.8104738154613467


In [8]:
# Save Models
models = [knn, nb, qda, lda]
for model in models:
    filename = str(model)[:str(model).find("(")] +"Model.pickle" #str(model) gets model name. .find("(") indexes name before parenthesis. Adds Model.pickle
    print(filename)
    pickle.dump(model, open(filename, "wb"))

KNeighborsClassifierModel.pickle
GaussianNBModel.pickle
QuadraticDiscriminantAnalysisModel.pickle
LinearDiscriminantAnalysisModel.pickle


In [27]:
for model in models:
    model_name = str(model)[:str(model).find("(")]
    train_predict = model.predict(x_train)
    train_result = metrics.accuracy_score(y_train, train_predict)
    test_predict = model.predict(x_test)
    test_result = metrics.accuracy_score(y_test, test_predict)
    print(f"{model_name} Train Accuracy: {train_result:.4f}")
    print(f"{model_name} Test Accuracy: {test_result:.4f}")

    train_confusion_matrix = pd.crosstab(y_train, train_predict)
    test_confusion_matrix = pd.crosstab(y_test, test_predict)
    print(f"{model_name} Train Confusion Matrix:\n{train_confusion_matrix}")
    print(f"{model_name} Test Confusion Matrix:\n{test_confusion_matrix}")

KNeighborsClassifier Train Accuracy: 0.9975
KNeighborsClassifier Test Accuracy: 0.9925
KNeighborsClassifier Train Confusion Matrix:
col_0    0    1
Label          
0      787    4
1        0  809
KNeighborsClassifier Test Confusion Matrix:
col_0    0    1
Label          
0      206    3
1        0  192
GaussianNB Train Accuracy: 0.9919
GaussianNB Test Accuracy: 0.9925
GaussianNB Train Confusion Matrix:
col_0    0    1
Label          
0      778   13
1        0  809
GaussianNB Test Confusion Matrix:
col_0    0    1
Label          
0      206    3
1        0  192
QuadraticDiscriminantAnalysis Train Accuracy: 0.9844
QuadraticDiscriminantAnalysis Test Accuracy: 0.9800
QuadraticDiscriminantAnalysis Train Confusion Matrix:
col_0    0    1
Label          
0      766   25
1        0  809
QuadraticDiscriminantAnalysis Test Confusion Matrix:
col_0    0    1
Label          
0      201    8
1        0  192
LinearDiscriminantAnalysis Train Accuracy: 0.8206
LinearDiscriminantAnalysis Test Accuracy: 